###***Catalog***

In [0]:
catalog=f"nyc_taxitrip"

###***Yellow Taxi Trip Combined table data***

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

silver_schema=f"{catalog}.silver"

# Get all tables from Silver schema
tables = spark.sql(f"SHOW TABLES IN {silver_schema}")

# Select only monthly yellow taxi trip tables
trip_tables = [row.tableName for row in tables.collect() if row.tableName.startswith("slv_yellow_tripdata_")]

print(f"Number of monthly tables: {len(trip_tables)}")
print(trip_tables)

# Read the first monthly table
df_gold = spark.read.table(f"{silver_schema}.{trip_tables[0]}")

# Combine the remaining monthly tables
for table_name in trip_tables[1:]:
    df_slv = spark.read.table(f"{silver_schema}.{table_name}")

    df_gold = df_gold.unionByName(df_slv,allowMissingColumns=True)

#Write the unified data to gold Schema
df_gold.write.option("mergeSchema", "true").mode("overwrite").saveAsTable(f"{catalog}.gold.gld_Yellow_taxi_trips")

df_gold.limit(20).display()


###***For Testing: Check the slv_tables along with their row count***

In [0]:
%sql
SELECT
    _source_file AS trip_month,
    COUNT(*) AS total_trips
FROM nyc_taxitrip.gold.gld_Yellow_taxi_trips
GROUP BY _source_file
ORDER BY trip_month;

###***Taxi Zone Lookup***

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

df_gld=spark.read.table(f"{catalog}.silver.slv_taxizone_lookup")

df_gld.write.option("mergeSchema", "true").mode("overwrite").saveAsTable(f"{catalog}.gold.gld_taxizone_lookup")
df_gld.limit(20).display()
